In [3]:
import pandas as pd
import numpy as np
import miner2.preprocess as prep
import miner2.coexpression as coex

In [4]:
# --- SECTION 1: PREPROCESSING DATA ---


#clinical = pd.read_csv('data_clinical_patient.txt', sep = '\t')
#print(clinical.shape)
# Gives 600 rows and 38 columns. First 4 rows are not patient data which means there are 596 patients. Fix: 

clinical = pd.read_csv('data_clinical_patient.txt', sep='\t', comment='#', skiprows=4)
#print(clinical.columns.tolist())

# Columns are  'OTHER_PATIENT_ID', 'PATIENT_ID', 
# ..., 'TISSUE_SOURCE_SITE', 'SITE_OF_TUMOR_TISSUE', 'OS_STATUS', 'OS_MONTHS', 'DFS_STATUS', 'DFS_MONTHS'


# The main data we need here is OS_STATUS and OS_MONTHS. These will be input to the Cox regression
# PATIENT_ID also needed to align with expression data

deceased = len(list(filter(lambda x: x == '1:DECEASED', list(clinical['OS_STATUS']))))
#print(deceased)
# 492/596 patients dead

# Copies the relevant columns, drops all NaN rows (>= 1 NaN val) and converts the string message for death into its integer value.
clinical_clean = clinical[['PATIENT_ID', 'OS_STATUS', 'OS_MONTHS']].copy()
clinical_clean = clinical[['PATIENT_ID', 'OS_STATUS', 'OS_MONTHS']].dropna(subset=['OS_STATUS', 'OS_MONTHS']).copy()
clinical_clean['OS_STATUS'] = (clinical_clean['OS_STATUS'] == '1:DECEASED').astype(int)

# Checking results:
#print(clinical_clean.shape) ; gives 437 patients * 3 columns
#print(clinical_clean.head()) ; data is showing as expected


# Now doing the same for expression data which is ALREADY Z-SCORED

expr = pd.read_csv('data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt', sep = '\t')


# df_expr_mrna_w_xxx , df_expr_wo_xxx

#print(expr.shape)
# 20531 genes in 168 patients. First 2 columns are Hugo Symbol and Entrez ID. Remaining 166 are patient IDs. 
# Set Hugo Symbol to index and drop Entrez ID. Since index isnt part of columns, only remaining column is patient_id
expr = expr.set_index('Hugo_Symbol').drop('Entrez_Gene_Id', axis=1, errors='ignore')


# We have 596 patients but only 166 with expression data
# Checking rna microarrays we get 403 patients with data
expr_arr = pd.read_csv('data_mrna_agilent_microarray_zscores_ref_all_samples.txt', sep='\t')
expr_arr = expr_arr.set_index('Hugo_Symbol').drop('Entrez_Gene_Id', axis=1, errors='ignore')


# Patient sets
clinical_patients = set(clinical_clean['PATIENT_ID'].tolist())                            # Gives a set of UNIQUE patient_ids
seq_patients = set([col[:12] for col in expr.columns if col.startswith('TCGA')])          # Extracts patient_ids in RNA seq data
arr_patients = set([col[:12] for col in expr_arr.columns if col.startswith('TCGA')])      # Extracts patient_ids in microarray data

# Taking all patient_ids that are common to the three sets
overlap = (seq_patients | arr_patients) & clinical_patients 

'''

RNA-seq patients: 160
Array patients: 401
Overlap with clinical: 439

'''

# Filter sequence and microarray data to overlapping primary tumor samples only (-01)
expr_filtered = expr[[col for col in expr.columns if col.endswith('-01') and col[:12] in overlap]]
expr_arr_filtered = expr_arr[[col for col in expr_arr.columns if col.endswith('-01') and col[:12] in overlap]]

#print(f"RNA-seq filtered: {expr_filtered.shape}")
#print(f"Array filtered: {expr_arr_filtered.shape}")

'''

RNA-seq filtered: (20531, 152)
Array filtered: (17814, 400)

'''

# Standardize column names
expr_filtered.columns = [col[:12] for col in expr_filtered.columns]
expr_arr_filtered.columns = [col[:12] for col in expr_arr_filtered.columns]

# Common genes: 14118
common_genes = expr_filtered.index.intersection(expr_arr_filtered.index)

# Filter sequence and microarray data to common genes
expr_seq_common = expr_filtered.loc[common_genes]
expr_arr_common = expr_arr_filtered.loc[common_genes]

# Patient splits
patients_in_both = seq_patients & arr_patients & clinical_patients
patients_seq_only = (seq_patients - arr_patients) & clinical_patients
patients_arr_only = (arr_patients - seq_patients) & clinical_patients

'''

Patients in both: 120
Patients seq only: 39
Patients arr only: 280

'''

# For patients in both, use array data
arr_both = expr_arr_common[list(patients_in_both & set(expr_arr_common.columns))]
seq_only = expr_seq_common[list(patients_seq_only & set(expr_seq_common.columns))]
arr_only = expr_arr_common[list(patients_arr_only & set(expr_arr_common.columns))]

# Drop duplicate genes
arr_both = arr_both[~arr_both.index.duplicated(keep='first')]
seq_only = seq_only[~seq_only.index.duplicated(keep='first')]
arr_only = arr_only[~arr_only.index.duplicated(keep='first')]

# Combine
expr_combined = pd.concat([arr_both, seq_only, arr_only], axis=1)

'''
Combined shape: (14118, 437)
Duplicate columns: 0
Duplicate genes: 0

'''

# Drop NaN gene names
expr_combined = expr_combined[expr_combined.index.notna()]

# Drop all-NaN rows and columns
expr_combined = expr_combined.dropna(how='all', axis=0)
expr_combined = expr_combined.dropna(how='all', axis=1)

# Align clinical to patients in expr_combined
clinical_clean = clinical_clean[clinical_clean['PATIENT_ID'].isin(expr_combined.columns)]
clinical_clean = clinical_clean.set_index('PATIENT_ID')

# Align expr_combined columns to clinical index order
expr_combined = expr_combined[clinical_clean.index]

'''

Expression: (14117, 437)
Clinical: (437, 2)

'''
# Counting NaN values (159200 of them)
#print(f"NaN values in matrix: {expr_combined.isna().sum().sum()}")

# Handling Nan: if < 20% values of a gene are NaN, replace it with mean in other patients
# Else: drop

# Calculate NaN percentage per gene

nan_pct = expr_combined.isna().sum(axis=1) / expr_combined.shape[1]

# Drop genes with >20% NaN
expr_combined = expr_combined[nan_pct <= 0.2]
#print(f"Genes after dropping high-NaN rows: {expr_combined.shape[0]}")

# Impute remaining NaNs with row mean
expr_combined = expr_combined.T.fillna(expr_combined.mean(axis=1)).T
#print(f"NaN values remaining: {expr_combined.isna().sum().sum()}")
#print(f"Final shape: {expr_combined.shape}") 

# Final shape: (13741, 437)

# Summary so far: expr_combined as combined expression data for 13741 genes across 437 patients. Priority order is microarray > rna-seq. 
# clinical_clean contains 437 patients with PATIENT_ID, OS_MONTHS, OS_STATUS. Both aligned to the same 12-character TCGA patient ID format

In [5]:
# --- SECTION 2: CLUSTERING GENES INTO REGULONS ---

import miner2.preprocess as prep
import miner2.coexpression as coex

import multiprocessing
#print(multiprocessing.cpu_count())

# Final cleanup
expr_clean = prep.removeNullRows(expr_combined)
expr_clean.columns = expr_clean.columns.astype(str)
expr_clean.index = expr_clean.index.astype(str)


#print(f"Final clean matrix: {expr_clean.shape}")
#print(f"NaN check: {expr_clean.isna().sum().sum()}")

# Decomposition on all 13.7k genes doesn't work. So we just take 8k genes w/ most variance

gene_variance = expr_clean.var(axis=1)
top_genes = gene_variance.nlargest(8000).index
expr_8k = expr_clean.loc[top_genes]

# Test run on all 14k genes
clusters = coex.cluster(expr_clean, numCores = 5)

2026-05-19 13:14:23 	 coexpression
2026-05-19 13:14:24 	 working on coexpression step 1 out of 5
2026-05-19 13:14:32 	 working on coexpression step 2 out of 5
2026-05-19 13:14:37 	 working on coexpression step 3 out of 5
2026-05-19 13:14:45 	 working on coexpression step 4 out of 5
2026-05-19 13:14:51 	 working on coexpression step 5 out of 5


In [6]:
# --- SECTION 3: DECOMPOSING REGULONS  ---

'''

Clusters found: 604 (with top 8k genes) and 1034 (with all genes)
Min: 7, Max: 250, Mean: 16.7

'''
print(len(clusters))

# Now decomposing clusters via PCA:

# Test decompose on one large real cluster
'''

Testing on cluster of size: 250
Decompose returned: 12 subgroups
Sizes: [70, 36, 19, 14, 11, 9, 11, 13, 7, 7, 7, 6]

'''

# For all clusters:

decomposed = []
total = len([c for c in clusters if len(c) >= 6])

for idx, cluster in enumerate(clusters):
    if len(cluster) >= 6:          # only consider clusters with atleast 6 genes; no point reducing smaller ones
        try:
            result = coex.decompose(cluster, expr_clean, minNumberGenes=6)
            if len(result) == 0:
                decomposed.append(cluster)
            else:
                decomposed.extend(result)
        except Exception as e:
            print(f"Skipping cluster of size {len(cluster)}: {e}")
        
        if idx % 50 == 0:
            print(f"Progress: {idx}/{total} clusters processed, {len(decomposed)} subgroups so far")

sizes = [len(c) for c in decomposed]


'''

After decompose: 663
Min: 6, Max: 70, Mean: 10.4

'''

1034
Progress: 0/1034 clusters processed, 17 subgroups so far
Progress: 50/1034 clusters processed, 122 subgroups so far
Progress: 100/1034 clusters processed, 199 subgroups so far
Progress: 150/1034 clusters processed, 259 subgroups so far
Progress: 200/1034 clusters processed, 314 subgroups so far
Progress: 250/1034 clusters processed, 365 subgroups so far
Progress: 300/1034 clusters processed, 415 subgroups so far
Progress: 350/1034 clusters processed, 465 subgroups so far
Progress: 400/1034 clusters processed, 515 subgroups so far
Progress: 450/1034 clusters processed, 565 subgroups so far
Progress: 500/1034 clusters processed, 615 subgroups so far
Progress: 550/1034 clusters processed, 665 subgroups so far
Progress: 600/1034 clusters processed, 715 subgroups so far
Progress: 650/1034 clusters processed, 765 subgroups so far
Progress: 700/1034 clusters processed, 815 subgroups so far
Progress: 750/1034 clusters processed, 865 subgroups so far
Progress: 800/1034 clusters processed, 

'\n\nAfter decompose: 663\nMin: 6, Max: 70, Mean: 10.4\n\n'

In [8]:
#print(f"\nDone!")
print(f"After decompose: {len(decomposed)}")
print(f"Min: {min(sizes)}, Max: {max(sizes)}, Mean: {sum(sizes)/len(sizes):.1f}")

After decompose: 1148
Min: 6, Max: 116, Mean: 10.3


In [9]:
# --- SECTION 4: RUNNING PCA TO VALIDATE REGULONS ---

from sklearn.decomposition import PCA

def validate_coexpression(cluster, expr, threshold=0.30):
    
    try:
        # Error here : consider a regulon with 12 genes of which 11 make it into expr_8k but 1 doesn't
        # Though this is a valid regulon it is not counted as the line below raises KeyError and doesn't add it to validated
        # Initial line: submatrix = expr.loc[cluster].T  # extracts only the genes in the given cluster
        # Fix: 
        
        valid_genes = [gene for gene in cluster if gene in expr.index]
        num_genes = len(valid_genes)
    
        # PCA requires at least 2 features (genes) to evaluate co-expression
        if num_genes < 2:
            return False

        try:
            # Extract submatrix. Since expr is (genes, patients), 
            # submatrix.T results in shape: (samples=patients, features=genes)
            submatrix = expr.loc[valid_genes].T 
        
            # Ensure we have at least 2 patients (samples)
            if submatrix.shape[0] < 2:
                return False
                
            # Fit 1-component PCA
            pca = PCA(n_components=1, random_state=42)
            pca.fit(submatrix)
            variance_explained = pca.explained_variance_ratio_[0] # returns a list and extracts PC1 which is its first value
            return variance_explained >= threshold
        
        except Exception:
        # Handles mathematical edge cases
            return False
            
        pca = PCA(n_components=1)                                   # does PC1 cause >= threshold variance?
        pca.fit(submatrix)
        
        return pca.explained_variance_ratio_[0] >= threshold
        
    except Exception as e:
        return False

validated = [c for c in decomposed if validate_coexpression(c, expr_clean)]

sizes = [len(c) for c in validated]



In [10]:
print(f"Before PCA validation: {len(decomposed)}")   
print(f"After PCA validation: {len(validated)}")    
print(f"Min: {min(sizes)}, Max: {max(sizes)}, Mean: {sum(sizes)/len(sizes):.1f}")

# Before PCA validation: 663
# After PCA validation: 505
# Min: 6, Max: 70, Mean: 9.9

Before PCA validation: 1148
After PCA validation: 899
Min: 6, Max: 116, Mean: 9.9


In [11]:
# --- SECTION 5: CALCULATING THE EIGENGENE SCORE FOR EACH REGULON


def get_eigengene(cluster, expr):                         # Args: cluster (regulon) and expr (df from which it takes seq data)
    
    try:
        
        # Filter down to the genes that are present in the expression matrix 
        # This is redundant here since validated only contains genes in expr
        valid_genes = [gene for gene in cluster if gene in expr.index]
        
        if len(valid_genes) < 2:                          # PCA needs >= 2 genes
            return None

        submatrix = expr.loc[valid_genes].T               # Extracts cluster genes, transposes shape to patients x genes
        pca = PCA(n_components=1, random_state=42)               
        scores = pca.fit_transform(submatrix)             # Fits PCA and get PC1 score for each patient
        eigen = pd.Series(                                # Convert to flattened Series with patient IDs as index
            scores.flatten(),
            index=submatrix.index                  
        )
        
        mean_expr = submatrix.mean(axis=1)                # Mean expression of cluster genes per patient
        if np.corrcoef(eigen, mean_expr)[0,1] < 0:        # .corrcoeff generates n*n matrix where offdiagonal elements are correlation coeffs
                                                          # Here correlation is symmetric i.e [0, 1] = [1,0]  
            
            eigen = -eigen                                # Flip sign (so that high exp relative to mean gives eigengene > 0)
            
        return eigen                                      # Returns eigengene value for a particular cluster
    
    except Exception as e:
        return None                              # Return None if anything goes wrong


# Compute eigen genes for all validated regulons

eigengenes = {}                               # Dict: regulon index (within the list validated) -> eigengene Series

for idx, cluster in enumerate(validated): 
    
    if idx % 100 == 0:                           # Print progress every 100 clusters
        
        print(f"Computing eigen gene {idx}/{len(validated)}...")
        
    eigen = get_eigengene(cluster, expr_clean)     # Compute eigengene for this cluster
    if eigen is not None:                        # Only store with cluster index as key if computation succeeded
        eigengenes[idx] = eigen                 

# Now eigengenes is a dict where a Series for each regulon is indexed by its position in the list validated
# Series contains the eigengene values in each of the 437 patients

Computing eigen gene 0/899...
Computing eigen gene 100/899...
Computing eigen gene 200/899...
Computing eigen gene 300/899...
Computing eigen gene 400/899...
Computing eigen gene 500/899...
Computing eigen gene 600/899...
Computing eigen gene 700/899...
Computing eigen gene 800/899...


In [12]:
# Replace [Not Available] with NaN in clinical_clean
clinical_clean = clinical_clean.replace('[Not Available]', np.nan)
clinical_clean['OS_MONTHS'] = pd.to_numeric(clinical_clean['OS_MONTHS'], errors='coerce')
clinical_clean['OS_STATUS'] = pd.to_numeric(clinical_clean['OS_STATUS'], errors='coerce')
clinical_clean = clinical_clean.dropna()

In [24]:
# --- SECTION 6: COX REGRESSION ON REGULONS TO IDENTIFY DISEASE-RELEVANCE ---

from lifelines import CoxPHFitter

def cox_regulon(eigen, clinical, duration_col='OS_MONTHS', event_col='OS_STATUS'):
    try:
        df = pd.DataFrame({
            'eigen': eigen,
            'duration': clinical[duration_col],
            'event': clinical[event_col]
        }).dropna()
        
        if len(df) < 10:
            return None, None
        
        cph = CoxPHFitter()
        cph.fit(df, duration_col='duration', event_col='event')
        
        hr = cph.hazard_ratios_['eigen']
        pval = cph.summary['p']['eigen']
        
        return hr, pval
    
    except Exception as e:
        return None, None

cox_results = {}
for idx, eigen in eigengenes.items():
    if idx % 100 == 0:
        print(f"Cox regression {idx}/{len(eigengenes)}...")
    
    hr, pval = cox_regulon(eigen, clinical_clean)
    if hr is not None:
        cox_results[idx] = {'hr': hr, 'pval': pval}

disease_regulons = {idx: res for idx, res in cox_results.items() if res['pval'] <= 0.05}

print(f"Total regulons tested: {len(cox_results)}")
print(f"Disease relevant regulons (p <= 0.05): {len(disease_regulons)}")

Cox regression 0/899...
Cox regression 100/899...
Cox regression 200/899...
Cox regression 300/899...
Cox regression 400/899...
Cox regression 500/899...
Cox regression 600/899...
Cox regression 700/899...
Cox regression 800/899...
Total regulons tested: 899
Disease relevant regulons (p <= 0.05): 181


In [18]:
# --- SECTION 7: QUICK CHECK USING KNOWN BIOLOGY ---

# Look at most significant regulons
cox_df = pd.DataFrame(cox_results).T
cox_df.columns = ['hr', 'pval']
cox_df = cox_df.sort_values('pval')

#print("Top 10 most significant regulons:")
#print(cox_df.head(10))

# High risk regulons (HR > 1)
high_risk = cox_df[cox_df['hr'] > 1]
low_risk = cox_df[cox_df['hr'] < 1]
#print(f"\nHigh risk regulons: {len(high_risk[high_risk['pval'] <= 0.05])}")       141 high risk
#print(f"Low risk regulons: {len(low_risk[low_risk['pval'] <= 0.05])}")            40 low risk

top_regulon_idx = cox_df.index[0]
#print(f"Top regulon index: {top_regulon_idx}")
#print(f"Genes: {validated[top_regulon_idx]}")

# Genes: ['CLDN7', 'TSPAN1', 'EREG', 'LRRC8E', 'RAB38', 'TFAP2C', 'SPINT1', 'HAL']
# The paper mentions EREG as a high-risk feature
# The regulon's HR = 1.1117 > 1 which matches this !!!

print(f"HR: {cox_df.loc[top_regulon_idx, 'hr']:.4f}")
print(f"p-val: {cox_df.loc[top_regulon_idx, 'pval']:.4e}")

HR: 1.1117
p-val: 1.4927e-07


In [19]:
# Pickle for using in other notebooks:

import pickle

with open('expr_clean.pkl', 'wb') as f:
    pickle.dump(expr_clean, f)

with open('validated.pkl', 'wb') as f:
    pickle.dump(validated, f)

with open('eigengenes.pkl', 'wb') as f:
    pickle.dump(eigengenes, f)

with open('cox_results.pkl', 'wb') as f:
    pickle.dump(cox_results, f)

print("All saved")

All saved


In [20]:
from scipy.stats import binomtest

def quantize_regulon_for_patient(patient_expr, regulon_genes):
    """
    patient_expr: pd.Series of all gene expressions for one patient (z-scored)
    regulon_genes: list of genes in the regulon
    """
    genes_in_data = [g for g in regulon_genes if g in patient_expr.index]
    if len(genes_in_data) < 3:
        return 0
    
    # Rank all genes for this patient
    n_genes = len(patient_expr)
    third = n_genes // 3
    
    sorted_genes = patient_expr.rank(method='average')
    
    # Which third does each regulon gene fall in?
    regulon_ranks = sorted_genes[genes_in_data]
    
    n = len(genes_in_data)
    
    # Count genes in upper third
    upper = (regulon_ranks > (2 * n_genes / 3)).sum()
    # Count genes in lower third  
    lower = (regulon_ranks < (n_genes / 3)).sum()
    
    p_upper = binomtest(int(upper), n, 1/3, alternative='greater').pvalue
    p_lower = binomtest(int(lower), n, 1/3, alternative='greater').pvalue
    
    if p_upper <= 0.05:
        return 1   # overactive
    elif p_lower <= 0.05:
        return -1  # underactive
    else:
        return 0   # neutral


In [30]:
# Setting up quantised matrix for the network via vectorised approach:

def quantize_matrix_fast(expr_df, disease_regulons, validated):
    """
    Vectorized quantization: for each patient, rank all genes once,
    then test all regulons against those ranks.
    """
    patients = expr_df.columns.tolist()
    n_genes = len(expr_df)
    
    # Precompute gene index lookup
    gene_to_idx = {g: i for i, g in enumerate(expr_df.index)}
    
    # Precompute regulon gene indices
    regulon_gene_idx = {}
    for reg_idx in disease_regulons:
        genes = validated[reg_idx]
        idx_list = [gene_to_idx[g] for g in genes if g in gene_to_idx]
        if len(idx_list) >= 3:
            regulon_gene_idx[reg_idx] = np.array(idx_list)
    
    valid_regulons = list(regulon_gene_idx.keys())
    n_regulons = len(valid_regulons)
    n_patients = len(patients)
    
    result = np.zeros((n_regulons, n_patients), dtype=np.int8)
    
    # Precompute thresholds
    lower_thresh = n_genes / 3
    upper_thresh = 2 * n_genes / 3
    
    expr_values = expr_df.values  # genes × patients numpy array
    
    for j in range(n_patients):
        if j % 50 == 0:
            print(f"Patient {j}/{n_patients}...")
        
        # Rank all genes for this patient in one shot
        ranks = pd.Series(expr_values[:, j]).rank(method='average').values
        
        for i, reg_idx in enumerate(valid_regulons):
            gene_idx = regulon_gene_idx[reg_idx]
            reg_ranks = ranks[gene_idx]
            n = len(reg_ranks)
            
            upper = int((reg_ranks > upper_thresh).sum())
            lower = int((reg_ranks < lower_thresh).sum())
            
            p_upper = binomtest(upper, n, 1/3, alternative='greater').pvalue
            p_lower = binomtest(lower, n, 1/3, alternative='greater').pvalue
            
            if p_upper <= 0.05:
                result[i, j] = 1
            elif p_lower <= 0.05:
                result[i, j] = -1
    
    quantized_matrix = pd.DataFrame(result, index=valid_regulons, columns=patients)
    return quantized_matrix

# Quantised matrix for all 899 validated regulons, not just 148 disease-relevant ones
quantized_matrix_1148 = quantize_matrix_fast(expr_clean, list(range(len(decomposed))), decomposed)

print(f"\nQuantized matrix full: {quantized_matrix_full.shape}")
print(pd.Series(quantized_matrix_full.values.flatten()).value_counts())

Patient 0/437...
Patient 50/437...
Patient 100/437...
Patient 150/437...
Patient 200/437...
Patient 250/437...
Patient 300/437...
Patient 350/437...
Patient 400/437...

Quantized matrix full: (1148, 437)
 0    324571
 1     92225
-1     84880
Name: count, dtype: int64


In [42]:
with open('quantized_matrix_1148.pkl', 'wb') as f:
    pickle.dump(quantized_matrix_1148, f)

with open('decomposed.pkl', 'wb') as f:
    pickle.dump(decomposed, f)

with open('cox_results.pkl', 'wb') as f:
    pickle.dump(cox_results, f)

with open('validated.pkl', 'wb') as f:
    pickle.dump(validated, f)

with open('clinical_clean.pkl', 'wb') as f:
    pickle.dump(clinical_clean, f)


In [36]:
print(f"\nQuantized matrix: {quantized_matrix_1148.shape}")
print(pd.Series(quantized_matrix_1148.values.flatten()).value_counts())



Quantized matrix: (1148, 437)
 0    324571
 1     92225
-1     84880
Name: count, dtype: int64
